# FLAG 2027 - Notebook 43a PROBE: do the clips contain a second speaker? (keynote-speaker front end, research round 10)

The FAME 2024 winner (HLT, 19.9 EER) ran a "keynote speaker" diarization front end that keeps only the main speaker's speech in every clip, in training and at test time. Their ablation credits it with about 2 points on the unheard-language cell. If MAV-Celeb v4 clips are interview videos with an interviewer's voice in them, all six of our voice encoders are embedding a mixture of two people, and the fix is a deterministic per-clip preprocessing step by a frozen public model (allowed by FAQ items 10 and 13 if declared; we would also ask the organizers).

This notebook only MEASURES whether the problem exists. It diarizes 300 random challenge-train clips and 300 random dev clips with `pyannote/speaker-diarization-3.1` (frozen, public) and reports, per split: the share of clips with more than one speaker, the share of audio spoken by the dominant speaker, and how often a second speaker holds more than 20% of the clip. No feature, model or score is changed.

**Before running:** on Hugging Face, with the account of your `HF_TOKEN`, open `pyannote/speaker-diarization-3.1` and `pyannote/segmentation-3.0` and accept the terms on both pages (one click each). Then: T4 GPU, `HF_TOKEN` in Colab Secrets, Runtime > Run all, about 20 minutes. Paste the **SUMMARY**.


In [ ]:
# Cell 1 [NB43a v1 6-Nov] - Drive, GPU (hard check), pyannote.audio, HF token (Colab Secrets only; never paste it anywhere)
from google.colab import drive, userdata
drive.mount('/content/drive')
!pip -q install "pyannote.audio>=3.1"
import torch, shutil
assert torch.cuda.is_available(), 'NO GPU in this runtime. Runtime > Change runtime type > T4 GPU > Save, then run again.'
dev = 'cuda'
HF_TOKEN = userdata.get('HF_TOKEN')
assert HF_TOKEN, 'HF_TOKEN not found in Colab Secrets (key icon on the left): add it and enable notebook access, then run again.'
print('[NB43a v1 6-Nov] Device:', torch.cuda.get_device_name(0), '| HF token found: True | disk free %.0f GB' % (shutil.disk_usage('/content')[2] / 1e9))


In [ ]:
# Cell 2 [NB43a v2 7-Nov] - diarize 300 challenge-train clips + 300 dev clips (frozen pyannote 3.1) and report how much second-speaker speech there is.
#          Measurement only: nothing is trained, no feature or score is produced. Per-clip results saved to results/nb43a_diarization.csv.
import os, json, time, zipfile, glob, random
import numpy as np, pandas as pd, soundfile as sf
from tqdm.auto import tqdm
BASE = '/content/drive/MyDrive'; PROJECT = os.path.join(BASE, 'FLAG2027'); RESDIR = os.path.join(PROJECT, 'results'); os.makedirs(RESDIR, exist_ok=True)
ZIP_DIR = json.load(open(os.path.join(PROJECT, 'paths.json')))['ZIP_DIR']
SUMMARY = []
def note(msg):
    print(msg); SUMMARY.append(msg)
def wavs_from_zip(zname, out):
    if not os.path.exists(out + '/done'):
        with zipfile.ZipFile(os.path.join(ZIP_DIR, zname)) as zf:
            zf.extractall(out, members=[m for m in zf.namelist() if m.lower().endswith('.wav')])
        open(out + '/done', 'w').close()
    return sorted(glob.glob(out + '/**/*.wav', recursive=True))
TR = wavs_from_zip('train_set.zip', '/content/d_train'); DV = wavs_from_zip('dev_set.zip', '/content/d_dev')
assert TR and DV, 'no wav files found in the zips - tell Claude'
N = 300; rng = np.random.default_rng(0)
SAMPLE = {'train': [TR[i] for i in rng.choice(len(TR), min(N, len(TR)), replace=False)], 'dev': [DV[i] for i in rng.choice(len(DV), min(N, len(DV)), replace=False)]}
note('clips: train %d (sampled %d) | dev %d (sampled %d)' % (len(TR), len(SAMPLE['train']), len(DV), len(SAMPLE['dev'])))
from pyannote.audio import Pipeline
try:
    PIPE = Pipeline.from_pretrained('pyannote/speaker-diarization-3.1', token=HF_TOKEN)
except TypeError:
    PIPE = Pipeline.from_pretrained('pyannote/speaker-diarization-3.1', use_auth_token=HF_TOKEN)
assert PIPE is not None, 'pyannote pipeline did not load: accept the terms of pyannote/speaker-diarization-3.1 AND pyannote/segmentation-3.0 on Hugging Face with the token account, then run again'
PIPE.to(torch.device(dev))
def diarize(path):
    info = sf.info(path); dur = info.duration
    out = PIPE(path)
    ann = getattr(out, 'speaker_diarization', out)      # pyannote.audio 4.x wraps the Annotation in a DiarizeOutput; 3.x returns it directly
    speech = {}
    for seg, _, spk in ann.itertracks(yield_label=True):
        speech[spk] = speech.get(spk, 0.0) + (seg.end - seg.start)
    tot = sum(speech.values()); top = max(speech.values()) if speech else 0.0
    second = sorted(speech.values())[-2] if len(speech) > 1 else 0.0
    return dict(dur=dur, n_spk=len(speech), speech=tot, dominant=top / tot if tot else float('nan'), second_share=second / tot if tot else 0.0, second_sec=second)
rows = []; t0 = time.time()
for split, paths in SAMPLE.items():
    for p in tqdm(paths, desc='diarize ' + split):
        try:
            r = diarize(p)
        except Exception as e:
            r = dict(dur=float('nan'), n_spk=-1, speech=float('nan'), dominant=float('nan'), second_share=float('nan'), second_sec=float('nan')); print('failed', p, repr(e)[:120])
        r.update(split=split, path=os.path.relpath(p, '/content')); rows.append(r)
R = pd.DataFrame(rows); R.to_csv(os.path.join(RESDIR, 'nb43a_diarization.csv'), index=False)
note('diarization done in %.0f min' % ((time.time() - t0) / 60))
for split in ['train', 'dev']:
    d = R[(R.split == split) & (R.n_spk >= 0)]
    note('%-5s | clips %d | duration median %.1f s (min %.1f, max %.1f) | speech share of clip median %.2f' % (split, len(d), d.dur.median(), d.dur.min(), d.dur.max(), (d.speech / d.dur).median()))
    note('      | speakers found: 0 or 1 -> %.1f%% | 2 -> %.1f%% | 3+ -> %.1f%%' % (100 * (d.n_spk <= 1).mean(), 100 * (d.n_spk == 2).mean(), 100 * (d.n_spk >= 3).mean()))
    note('      | dominant speaker share of speech: median %.2f | clips where a second speaker holds > 20%% of the speech: %.1f%% | > 35%%: %.1f%% | second speaker > 3 s: %.1f%%' % (d.dominant.median(), 100 * (d.second_share > 0.20).mean(), 100 * (d.second_share > 0.35).mean(), 100 * (d.second_sec > 3).mean()))
multi = R[(R.n_spk >= 2)]
if len(multi):
    note('worst 5 clips (largest second-speaker share): ' + '; '.join('%s %.2f' % (os.path.basename(r.path), r.second_share) for r in multi.sort_values('second_share', ascending=False).head(5).itertuples()))
share20 = 100 * (R[R.n_spk >= 0].second_share > 0.20).mean()
note('VERDICT: %s' % ('a keynote-speaker front end is WORTH a night (>= 15%% of clips carry a real second speaker) -> Claude builds NB43' if share20 >= 15 else 'second speakers are rare (< 15%% of clips) -> the front end would not pay for its cost; closed'))
print('=' * 70); print('SUMMARY  (copy everything between the lines and paste to Claude)'); print('=' * 70)
for s in SUMMARY:
    print(s)
print('=' * 70)
